In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank, dense_rank
data = [
(1, "Ravi Kumar", "Engineering", "Hyderabad", 92000),
(2, "Sneha Reddy", "Engineering", "Bengaluru", 97000),
(3, "Amit Shah", "Finance", "Mumbai", 88000),
(4, "Pooja Nair", "HR", "Chennai", 69000),
(5, "Nikhil Verma", "Engineering", "Delhi", 85000),
(6, "Meera Iyer", "Finance", "Hyderabad", 91000),
(7, "Karan Malhotra", "Marketing", "Mumbai", 76000),
(8, "Anjali Rao", "HR", "Bengaluru", 72000),
(9, "Vivek Joshi", "Finance", "Delhi", 94000),
(10, "Fatima Khan", "Marketing", "Chennai", 81000),
(11, "Aditya Menon", "Engineering", "Mumbai", 99000),
(12, "Lakshmi Das", "Finance", "Bengaluru", 87000)
]
columns = ["emp_id", "emp_name", "department", "city", "salary"]
df = spark.createDataFrame(data, columns)
display(df)

emp_id,emp_name,department,city,salary
1,Ravi Kumar,Engineering,Hyderabad,92000
2,Sneha Reddy,Engineering,Bengaluru,97000
3,Amit Shah,Finance,Mumbai,88000
4,Pooja Nair,HR,Chennai,69000
5,Nikhil Verma,Engineering,Delhi,85000
6,Meera Iyer,Finance,Hyderabad,91000
7,Karan Malhotra,Marketing,Mumbai,76000
8,Anjali Rao,HR,Bengaluru,72000
9,Vivek Joshi,Finance,Delhi,94000
10,Fatima Khan,Marketing,Chennai,81000


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank, avg, when
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
df_rank = df.withColumn("salary_rank", rank().over(window_spec))
df_rank.show()

+------+--------------+-----------+---------+------+-----------+
|emp_id|      emp_name| department|     city|salary|salary_rank|
+------+--------------+-----------+---------+------+-----------+
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|          1|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|          2|
|     1|    Ravi Kumar|Engineering|Hyderabad| 92000|          3|
|     5|  Nikhil Verma|Engineering|    Delhi| 85000|          4|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|          1|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|          2|
|     3|     Amit Shah|    Finance|   Mumbai| 88000|          3|
|    12|   Lakshmi Das|    Finance|Bengaluru| 87000|          4|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|          1|
|     4|    Pooja Nair|         HR|  Chennai| 69000|          2|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|          1|
|     7|Karan Malhotra|  Marketing|   Mumbai| 76000|          2|
+------+--------------+--

In [0]:
df = df.withColumn("salary_level",when(col("salary") >= 95000, "Very High").when(col("salary") >= 85000, "High").otherwise("Standard"))
df.show()

+------+--------------+-----------+---------+------+------------+
|emp_id|      emp_name| department|     city|salary|salary_level|
+------+--------------+-----------+---------+------+------------+
|     1|    Ravi Kumar|Engineering|Hyderabad| 92000|        High|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|   Very High|
|     3|     Amit Shah|    Finance|   Mumbai| 88000|        High|
|     4|    Pooja Nair|         HR|  Chennai| 69000|    Standard|
|     5|  Nikhil Verma|Engineering|    Delhi| 85000|        High|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|        High|
|     7|Karan Malhotra|  Marketing|   Mumbai| 76000|    Standard|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|    Standard|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|        High|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|    Standard|
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|   Very High|
|    12|   Lakshmi Das|    Finance|Bengaluru| 87000|        High|
+------+--

In [0]:
df_ranked = df.withColumn("rnk", rank().over(window_spec))
top2 = df_ranked.filter(col("rnk") <= 2)
top2.show()

+------+--------------+-----------+---------+------+------------+---+
|emp_id|      emp_name| department|     city|salary|salary_level|rnk|
+------+--------------+-----------+---------+------+------------+---+
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|   Very High|  1|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|   Very High|  2|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|        High|  1|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|        High|  2|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|    Standard|  1|
|     4|    Pooja Nair|         HR|  Chennai| 69000|    Standard|  2|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|    Standard|  1|
|     7|Karan Malhotra|  Marketing|   Mumbai| 76000|    Standard|  2|
+------+--------------+-----------+---------+------+------------+---+



In [0]:
df_avg = df.groupBy("department").agg(avg("salary").alias("avg_salary"))
df_avg.show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|   93250.0|
|    Finance|   90000.0|
|         HR|   70500.0|
|  Marketing|   78500.0|
+-----------+----------+

